## Unit 11. 傅立葉轉換與頻譜分析 課堂作業
- 課程編號: CHE3502
- 學年度: 114下
- 上課時間: 每週一 16:10-19:20
-
- 指導教授: ＯＯＯ 教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit11_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit11'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# scipy.fft — 本課程主要工具
from scipy.fft import (
    fft, ifft, rfft, irfft,
    fftfreq, rfftfreq, fftshift, ifftshift,
    next_fast_len
)

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
### **I. 練習題：化工製程感測器訊號之頻譜分析**

某化工廠蒸餾塔底部安裝了壓力感測器，以 $f_s = 500\;\text{Hz}$ 的取樣頻率記錄塔底壓力訊號，總記錄時間 $T = 10\;\text{s}$。

已知該系統可能存在以下振動來源：
- **再沸器循環泵**：轉速 1800 RPM，葉片數 6 → 葉片通過頻率 (BPF) = $1800 \times 6 / 60 = 180\;\text{Hz}$
- **塔板液泛週期性脈動**：約 $8\;\text{Hz}$
- **進料管線機械振動**：約 $50\;\text{Hz}$

請使用下方提供的模擬訊號，完成以下頻譜分析任務。

In [ ]:
# ========================================
# 模擬蒸餾塔壓力感測器訊號（請勿修改此 Cell）
# ========================================
np.random.seed(42)

fs = 500          # 取樣頻率 [Hz]
T  = 10.0         # 記錄時間 [s]
N  = int(fs * T)  # 總取樣點數
t  = np.linspace(0, T, N, endpoint=False)

# --- 訊號成分 ---
# 1. 塔板液泛脈動：8 Hz, 振幅 3.0
x_flood = 3.0 * np.sin(2 * np.pi * 8 * t)

# 2. 進料管線機械振動：50 Hz, 振幅 1.5
x_pipe = 1.5 * np.sin(2 * np.pi * 50 * t + np.pi/4)

# 3. 再沸器泵浦 BPF：180 Hz, 振幅 0.8 + 二次諧波 360 Hz (超過奈奎斯特頻率!)
#    注意：360 Hz > fs/2 = 250 Hz，此成分會產生混疊
x_pump = 0.8 * np.sin(2 * np.pi * 180 * t)

# 4. 高頻成分（超出奈奎斯特）：360 Hz, 振幅 0.4
x_alias = 0.4 * np.sin(2 * np.pi * 360 * t)

# 5. DC 偏移（壓力基線）+ 線性漂移
dc_drift = 101.3 + 0.5 * t  # 基線 101.3 kPa，漂移 0.5 kPa/s

# 6. 高斯隨機雜訊
noise = 0.6 * np.random.randn(N)

# 合成訊號
x_raw = dc_drift + x_flood + x_pipe + x_pump + x_alias + noise

print(f"✓ 模擬訊號已生成")
print(f"  取樣頻率: {fs} Hz")
print(f"  記錄時間: {T} s")
print(f"  取樣點數: {N}")
print(f"  奈奎斯特頻率: {fs/2} Hz")
print(f"\n  訊號成分:")
print(f"    塔板液泛脈動:   8 Hz,  A = 3.0")
print(f"    進料管線振動:  50 Hz,  A = 1.5")
print(f"    再沸器泵浦 BPF: 180 Hz, A = 0.8")
print(f"    二次諧波(混疊): 360 Hz, A = 0.4 → 混疊至 {abs(360 - fs)} Hz")
print(f"    DC 偏移 + 線性漂移")
print(f"    高斯雜訊: σ = 0.6")

### **1.1 訊號前處理與時域觀察**
- 繪製原始訊號 `x_raw` 的時域波形圖（顯示前 1 秒即可）
- 對訊號進行**去趨勢**處理（移除 DC 偏移與線性漂移），使用 `np.polyfit()` 一次多項式
- 繪製去趨勢後的訊號波形圖，比較前後差異
- 說明為什麼頻譜分析前需要去趨勢處理

### **1.2 單邊幅度頻譜計算**
- 使用 `scipy.fft.rfft()` 計算去趨勢後訊號的單邊幅度頻譜
- 正確進行幅度歸一化（DC 分量、一般頻率、奈奎斯特頻率分別處理）
- 使用 `scipy.fft.rfftfreq()` 計算對應的頻率軸
- 繪製幅度頻譜圖（頻率範圍 0 ~ 250 Hz）
- 標註所有可識別的峰值頻率，並記錄其振幅
- 分析：360 Hz 的訊號成分在頻譜中出現在哪個頻率？為什麼？（混疊現象分析）

### **1.3 視窗函數效果比較**
- 分別使用 **Rectangular**、**Hann**、**Hamming**、**Blackman** 四種視窗函數
- 對去趨勢後的訊號套用視窗，計算單邊幅度頻譜（注意幅度修正因子 CG）
- 以 dB 為單位（ $20\log_{10}(\text{amplitude})$ ）繪製四種視窗的頻譜比較圖（子圖 2×2）
- 將頻率範圍聚焦於 40 ~ 60 Hz，觀察 50 Hz 峰值的旁葉抑制效果
- 分析：哪種視窗在本訊號中表現最佳？為什麼？

### **1.4 功率頻譜密度（PSD）估計**
- 使用 `scipy.fft.rfft()` 手動實作 **Welch 法**（分段長度 `nperseg=512`，50% 重疊，Hann 視窗）
- 繪製 **週期圖法** 與 **Welch 法** 的 PSD 比較圖（使用 `semilogy` 對數刻度）
- 使用 **Parseval 定理** 驗證 PSD 積分結果是否等於訊號均方功率
- 印出驗證結果：時域均方功率、PSD 積分功率、相對誤差百分比
- 分析：Welch 法相較於週期圖法有什麼優勢？

### **1.5 自相關函數（ACF）與週期估計**
- 使用 **Wiener-Khinchin 定理**（FFT 方法）計算去趨勢訊號的自相關函數（ACF）

$$
R_x[m] = \text{IFFT}\bigl(|X[k]|^2\bigr)
$$

- 使用 `scipy.fft.next_fast_len()` 進行零填充以提升計算效率
- 繪製歸一化 ACF 曲線（正延遲部分，顯示 0 ~ 0.5 秒）
- 從 ACF 峰值間距估計訊號的主要週期與對應頻率
- 將 ACF 估計結果與幅度頻譜的峰值頻率進行交叉驗證

### **1.6 結果整理與分析**
- 建立一個結果摘要表格（使用 `print` 輸出），包含：

| 頻率成分 | 理論頻率 (Hz) | 理論振幅 | 頻譜量測頻率 (Hz) | 頻譜量測振幅 | 備註 |
|---------|-------------|---------|----------------|-----------|----|
| 塔板液泛 | 8 | 3.0 | ? | ? | |
| 管線振動 | 50 | 1.5 | ? | ? | |
| 泵浦 BPF | 180 | 0.8 | ? | ? | |
| 二次諧波 | 360 | 0.4 | ? | ? | 混疊 |

- 分析各頻率成分的量測振幅與理論振幅是否一致，解釋可能的誤差來源
- 說明如何避免 360 Hz 的混疊問題（從取樣頻率的角度提出建議）

---
## II. 額外加分作業 (自由繳交)
- 學生可自由選擇是否繳交加分作業. (下週上課前完成email電子檔)
- 分數會加在最後總成績上, 每份作業加0.1 ~ 1.0分.
- 分數加的不多, 真的不一定要交加分作業, 正常出席上課做好期末報告即可.
- 加分作業不一定要全部都做完, 想繳交至少完成其中一項實驗即可.
- 請務必自行完成! 你可以問AI, 問同學, 問學長姊, 甚至參考以前別人的作業, 但一定要自行[吸收][執行][整理]結果!
- 不要貼AI的答案寄給老師看, 你自己看就好.
- 如果系統自動比對發現作業內容(與他人雷同, 原創性低, 抄襲比例過高), 作業的分數會0分.
- 如果你直接100%複製別人的作業繳交, 你會直接被當掉(提供作業給別人複製的也一樣).
- 老師看你作業要花時間, 真的有心想寫加分作業, 請好好自己做.

---
### 使用課堂作業所提供的模擬訊號
- 可嘗試修改模擬訊號的參數（改變頻率、振幅、增加成分等）
- 產生獨一無二屬於你的訊號資料
- 然後開始進行下面實驗

### **實驗1: STFT 時頻分析 — 視窗長度影響**

產生一段**非穩態訊號**（例如：前 5 秒包含 20 Hz，後 5 秒切換為 60 Hz，中間有過渡區），並使用手動 STFT（`scipy.fft.rfft()` 滑動視窗法）分析。

比較以下 4 種視窗長度：
- **短視窗**: `window_len = 64`
- **中視窗**: `window_len = 128`
- **長視窗**: `window_len = 256`
- **超長視窗**: `window_len = 512`

**要求**:
- 使用 `pcolormesh` 繪製 4 個 STFT 頻譜圖（子圖 2×2）
- 記錄每種視窗長度對應的頻率解析度 $\Delta f = f_s / L$ 與時間解析度 $\Delta t = L / f_s$
- **分析**: 說明時頻取捨（time-frequency trade-off）現象

### **實驗2: Welch 法參數影響 — 分段長度與重疊率**

使用練習題一提供的模擬訊號（去趨勢後），比較不同 Welch 參數對 PSD 估計的影響。

比較以下 4 種分段長度:
- `nperseg = 128`
- `nperseg = 256`
- `nperseg = 512`
- `nperseg = 1024`

**要求**:
- 固定 50% 重疊率，Hann 視窗
- 繪製 4 種分段長度的 PSD 比較圖（疊在同一張圖上）
- 記錄每種分段長度的頻率解析度 $\Delta f$ 與分段數量
- **分析**: 分段長度如何影響 PSD 的平滑度與頻率解析度？

### **實驗3: 取樣頻率與混疊現象**

自行產生一個包含 **200 Hz** 成分的訊號，分別以下列 4 種取樣頻率進行取樣：
- `fs = 800` Hz（充足取樣）
- `fs = 500` Hz（充足取樣）
- `fs = 350` Hz（不足取樣）
- `fs = 250` Hz（不足取樣）

**要求**:
- 對每種取樣頻率計算幅度頻譜
- 繪製 4 張頻譜圖（子圖 2×2），標註 200 Hz 的理論位置與實際觀測到的峰值位置
- 計算每種取樣頻率下的混疊頻率 $f_{\mathrm{alias}} = |f_{\mathrm{signal}} - k \cdot f_s|$
- **分析**: 為什麼工業上建議取樣頻率至少為目標頻率的 5~10 倍（而非僅 2 倍）？

### **實驗4: 訊號長度與頻率解析度**

產生一個包含兩個**密集頻率**成分的訊號： $f_1 = 50\;\text{Hz}$ 與 $f_2 = 52\;\text{Hz}$（相差僅 2 Hz），取樣頻率固定 $f_s = 500\;\text{Hz}$。

比較以下 4 種記錄時間：
- $T = 0.2\;\text{s}$ → $\Delta f = 5.0\;\text{Hz}$
- $T = 0.5\;\text{s}$ → $\Delta f = 2.0\;\text{Hz}$
- $T = 1.0\;\text{s}$ → $\Delta f = 1.0\;\text{Hz}$
- $T = 2.0\;\text{s}$ → $\Delta f = 0.5\;\text{Hz}$

**要求**:
- 對每種記錄時間計算幅度頻譜（使用 Hann 視窗）
- 繪製 4 張頻譜圖（子圖 2×2），頻率範圍 40 ~ 60 Hz
- 觀察在哪種記錄時間下能清楚分辨兩個頻率峰值
- **分析**: 頻率解析度 $\Delta f = 1/T$ 的物理意義是什麼？若要分辨兩個相差 $\Delta$ Hz 的頻率，至少需要多長的記錄時間？

### 💭 思考題

1. 為什麼在進行 FFT 頻譜分析前，通常需要先對訊號進行去均值（去 DC）與去趨勢處理？如果不做，頻譜會有什麼問題？
2. 視窗函數（如 Hann、Hamming）如何抑制頻譜洩漏？使用視窗函數後，為什麼幅度需要乘以修正因子 $1/\text{CG}$ ？
3. 取樣定理要求 $f_s \geq 2f_{\max}$，但為什麼工業上通常建議 $f_s \geq 5 \sim 10 \cdot f_{\max}$ ？
4. Welch 法為什麼比週期圖法更適合估計 PSD？「50% 重疊」在其中扮演什麼角色？
5. 如果一段訊號的頻率隨時間變化（非穩態），標準 FFT 的結果會有什麼問題？STFT 如何解決這個問題？
6. 自相關函數（ACF）與功率頻譜密度（PSD）之間的 Wiener-Khinchin 定理是什麼？為什麼用 FFT 計算 ACF 比直接計算更有效率？
7. 在化工製程監控中，頻譜分析可以用來檢測哪些異常情況？請舉出至少兩個具體應用場景。

---
# 想對老師說的話